In [0]:
# === J12 - MLflow Model Registry with Unity Catalog aliases ===
%pip install mlflow xgboost scikit-learn -q
dbutils.library.restartPython()

In [0]:
import sys, os
repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

import importlib
for mod in ['src.ingestion', 'src.cleaning', 'src.transformation',
            'src.ml.train', 'src.ml.mlflow_utils', 'src.ml.evaluate']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("j12-registry").getOrCreate()
spark.sql("USE CATALOG workspace")

client = MlflowClient()
print(f"✓ MLflow: {mlflow.__version__}, Spark: {spark.version}")

In [0]:
# === Current Model Registry state ===
print("=" * 60)
print("Current Model Registry State")
print("=" * 60)

registered_models = client.search_registered_models()

if registered_models:
    for rm in registered_models:
        print(f"\n--- Model: {rm.name} ---")
        
        # Get versions
        versions = client.search_model_versions(f"name='{rm.name}'")
        if versions:
            print(f"  Versions ({len(versions)} total):")
            for v in sorted(versions, key=lambda x: int(x.version)):
                print(f"    Version {v.version}: status={v.status}, run_id={v.run_id[:12]}...")
        
        # Get aliases (Unity Catalog uses aliases, not stages)
        try:
            aliases = client.get_registered_model_aliases(rm.name)
            if aliases:
                print(f"  Aliases:")
                for alias in aliases:
                    print(f"    '{alias.alias}' → version {alias.version}")
        except Exception:
            print(f"  Aliases: (not available or none)")
else:
    print("No registered models found.")

In [0]:
# === Register v2 = rf_config_1 (second best temporal F1) ===
print("=" * 60)
print("Registering Version 2 (rf_config_1)")
print("=" * 60)

# Find the experiment
for exp_name in [
    "/Users/salmacharki721@gmail.com/mlflow-experiments/cic-ids2017-intrusion-detection",
    "/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline/cic-ids2017-intrusion-detection",
]:
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp:
        break

runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])

# Filter to dual-eval runs (with training_split tag)
if 'tags.training_split' in runs.columns:
    dual_runs = runs[runs['tags.training_split'].notna()].copy()
else:
    dual_runs = runs.copy()

# Drop duplicate run names
if 'tags.mlflow.runName' in dual_runs.columns:
    dual_runs = dual_runs.drop_duplicates(subset='tags.mlflow.runName', keep='first')

# === EXPLICIT selection of rf_config_1 (not by position) ===
v2_run = dual_runs[dual_runs['tags.mlflow.runName'] == 'rf_config_1']

if v2_run.empty:
    print("⚠️ rf_config_1 not found. Available runs:")
    print(dual_runs['tags.mlflow.runName'].tolist())
else:
    v2_run = v2_run.iloc[0]
    v2_run_id = v2_run['run_id']
    v2_temp_f1 = v2_run.get('metrics.temporal_f1_score', 'N/A')
    
    print(f"  Selected: rf_config_1")
    print(f"  Run ID: {v2_run_id}")
    print(f"  Temporal F1: {v2_temp_f1}")
    
    # Register as version 2
    model_name = "cic-ids2017-intrusion-detection-best"
    model_uri = f"runs:/{v2_run_id}/model"
    
    try:
        result = mlflow.register_model(model_uri=model_uri, name=model_name)
        print(f"\n✓ Registered as '{model_name}' version {result.version}")
    except Exception as e:
        print(f"\nRegistration: {e}")

In [0]:
# === Assign aliases based on RUN ID (not version position) ===
print("=" * 60)
print("Assigning Aliases by Run ID (Unity Catalog)")
print("=" * 60)

model_name = "workspace.default.cic-ids2017-intrusion-detection-best"

# === Run IDs from J11 results ===
# rf_config_2 = best temporal F1 (0.583) → champion
# rf_config_1 = second best temporal F1 (0.571) → candidate
rf2_run_id = "f2a34e1540c34006a619cb139c8e24d7"  # rf_config_2
rf1_run_id = "0d5ea319c8ad450abe449701d7c1e5ab"  # rf_config_1

# Get all versions
versions = client.search_model_versions(f"name='{model_name}'")

print(f"\n--- All versions ---")
for v in sorted(versions, key=lambda x: int(x.version)):
    print(f"  Version {v.version}: run_id={v.run_id[:12]}...")

# === Find versions by Run ID ===
champion_version = None
candidate_version = None

for v in versions:
    if v.run_id == rf2_run_id:
        champion_version = int(v.version)
    elif v.run_id == rf1_run_id:
        candidate_version = int(v.version)

print(f"\n--- Found by Run ID ---")
print(f"  rf_config_2 (run_id={rf2_run_id[:12]}...) → version {champion_version}")
print(f"  rf_config_1 (run_id={rf1_run_id[:12]}...) → version {candidate_version}")

if champion_version is None:
    raise ValueError("rf_config_2 run not found in Model Registry")
if candidate_version is None:
    raise ValueError("rf_config_1 run not found in Model Registry")

# === Assign aliases ===
print(f"\n--- Setting 'champion' alias → version {champion_version} (rf_config_2) ---")
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=champion_version,
)
print(f"✓ 'champion' → v{champion_version} (rf_config_2, temporal F1=0.583)")

print(f"\n--- Setting 'candidate' alias → version {candidate_version} (rf_config_1) ---")
client.set_registered_model_alias(
    name=model_name,
    alias="candidate",
    version=candidate_version,
)
print(f"✓ 'candidate' → v{candidate_version} (rf_config_1, temporal F1=0.571)")

# === Verify aliases using get_model_version_by_alias ===
print(f"\n--- Verifying aliases ---")
try:
    champion_info = client.get_model_version_by_alias(model_name, "champion")
    print(f"  champion → v{champion_info.version}, run_id={champion_info.run_id[:12]}...")
except Exception as e:
    print(f"  champion verification: {e}")

try:
    candidate_info = client.get_model_version_by_alias(model_name, "candidate")
    print(f"  candidate → v{candidate_info.version}, run_id={candidate_info.run_id[:12]}...")
except Exception as e:
    print(f"  candidate verification: {e}")

In [0]:
# === Load models by alias ===
print("=" * 60)
print("Loading Models by Alias")
print("=" * 60)

model_name = "workspace.default.cic-ids2017-intrusion-detection-best"
print("\n--- Loading 'champion' (v2, rf_config_2) ---")
try:
    model_champion = mlflow.sklearn.load_model(f"models:/{model_name}@champion")
    print(f"✓ Loaded champion: {type(model_champion).__name__}")
except Exception as e:
    print(f"Error: {e}")
    model_champion = None

print("\n--- Loading 'candidate' (v4, rf_config_1) ---")
try:
    model_candidate = mlflow.sklearn.load_model(f"models:/{model_name}@candidate")
    print(f"✓ Loaded candidate: {type(model_candidate).__name__}")
except Exception as e:
    print(f"Error: {e}")
    model_candidate = None

In [0]:
# === Offline A/B-style comparison (NOT a production A/B test) ===
print("=" * 60)
print("Offline A/B-style Comparison: champion vs candidate")
print("=" * 60)
print("(This is an OFFLINE comparison on the same test set,")
print(" not a production A/B test with real traffic.)\n")

from src.ml.evaluate import prepare_temporal_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Prepare temporal test data
print("--- Preparing temporal test data ---")
X_train_temp, y_train_temp, X_test_temp, y_test_temp, feature_names, test_metadata = (
    prepare_temporal_split(spark, random_state=42)
)

# Align columns
champion_features = model_champion.feature_names_in_
for col in set(champion_features) - set(X_test_temp.columns):
    X_test_temp[col] = 0
X_test_temp = X_test_temp[champion_features]

# Predictions
print("\n--- Getting predictions ---")
y_pred_champion = model_champion.predict(X_test_temp)
y_pred_candidate = model_candidate.predict(X_test_temp)

# Metrics
metrics_champion = {
    'accuracy': accuracy_score(y_test_temp, y_pred_champion),
    'precision': precision_score(y_test_temp, y_pred_champion),
    'recall': recall_score(y_test_temp, y_pred_champion),
    'f1': f1_score(y_test_temp, y_pred_champion),
}

metrics_candidate = {
    'accuracy': accuracy_score(y_test_temp, y_pred_candidate),
    'precision': precision_score(y_test_temp, y_pred_candidate),
    'recall': recall_score(y_test_temp, y_pred_candidate),
    'f1': f1_score(y_test_temp, y_pred_candidate),
}

print("\n--- Comparison results ---")
comparison = pd.DataFrame([
    {'Version': 'champion (v2, rf_config_2)', **metrics_champion},
    {'Version': 'candidate (v4, rf_config_1)', **metrics_candidate},
])
display(comparison)

# Prediction agreement
agreement = (y_pred_champion == y_pred_candidate).mean()
print(f"\nPrediction agreement: {agreement:.4f} ({agreement*100:.1f}%)")
print(f"Disagreement: {(1-agreement)*100:.1f}% of test samples")

if metrics_champion['f1'] > metrics_candidate['f1']:
    print(f"\n✓ champion (v2) performs better → 'candidate' stays as comparison version")
elif metrics_champion['f1'] < metrics_candidate['f1']:
    print(f"\n⚠️ candidate (v4) performs better → consider reassigning 'champion' to v2")
else:
    print(f"\n= Both versions perform identically")

In [0]:
# === Pedagogical routing simulation (NOT real production traffic routing) ===
print("=" * 60)
print("Pedagogical Routing Simulation")
print("=" * 60)
print("(This is a SIMULATION for educational purposes,")
print(" not a real production traffic routing system.)\n")

# Simulate 1000 "requests" split between champion (70%) and candidate (30%)
np.random.seed(42)
n_requests = 1000
routing = np.random.choice(['champion', 'candidate'], size=n_requests, p=[0.7, 0.3])

print(f"--- Simulated traffic distribution ---")
print(f"  champion (70%):  {sum(routing == 'champion')} requests")
print(f"  candidate (30%): {sum(routing == 'candidate')} requests")

# Sample 1000 rows from test set
sample_indices = np.random.choice(len(X_test_temp), size=min(n_requests, len(X_test_temp)), replace=False)
X_sample = X_test_temp.iloc[sample_indices]
y_sample = y_test_temp.iloc[sample_indices]

# Route each "request" to the assigned model
print(f"\n--- Routing {len(X_sample)} requests ---")
predictions = []
for i in range(len(X_sample)):
    if routing[i] == 'champion':
        pred = model_champion.predict(X_sample.iloc[[i]])[0]
    else:
        pred = model_candidate.predict(X_sample.iloc[[i]])[0]
    predictions.append(pred)

predictions = np.array(predictions)

# Overall metrics
accuracy = accuracy_score(y_sample, predictions)
f1 = f1_score(y_sample, predictions)

print(f"\n--- Simulated routing performance ---")
print(f"  Overall accuracy: {accuracy:.4f}")
print(f"  Overall F1-score: {f1:.4f}")
print(f"\nNote: In a real production system, you would:")
print(f"  1. Monitor per-version metrics in real-time")
print(f"  2. Gradually shift traffic to the better-performing version")
print(f"  3. Use shadow deployment before A/B testing")
print(f"  4. Implement rollback if metrics degrade")

# J12 — MLflow Model Registry with Unity Catalog Aliases

## Objective

This phase demonstrates model versioning and lifecycle management using the
**MLflow Model Registry with Unity Catalog aliases**.

The objective is not to build a production deployment, but to practice
model management and MLOps concepts on the models evaluated in J11.

---

## 1. Model Versioning

Two previously evaluated Random Forest configurations from J11 were selected:

| Role | Configuration | Registry Version | Temporal F1 |
|------|---------------|------------------|-------------|
| Champion | `rf_config_2` | v2 | 0.5829 |
| Candidate | `rf_config_1` | v4 | 0.5711 |

The versions were identified explicitly using their **MLflow Run IDs**, rather
than assuming that registry version numbers correspond to model performance.

This is important because additional model versions may already exist in the
registry.

---

## 2. Unity Catalog Aliases

The following aliases were assigned:

- `champion` → **v2 (`rf_config_2`)**
- `candidate` → **v4 (`rf_config_1`)**

The aliases were verified using `get_model_version_by_alias()`.

This approach allows a logical name such as `champion` or `candidate` to point
to a specific model version without depending on the numerical version ID.

The `champion` alias can therefore be reassigned to another version later
without changing the version history.

---

## 3. Loading Models by Alias

Both models were successfully loaded using their Unity Catalog aliases:

```text
models:/workspace.default.cic-ids2017-intrusion-detection-best@champion
models:/workspace.default.cic-ids2017-intrusion-detection-best@candidate